In [ ]:
# Import necessary libraries
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import harmonypy as hm
import seaborn as sns
from copy import copy
import os
import scvi

from pyensembl import EnsemblRelease

# Import lab's toolkit
from labcore import scrnaseq

In [ ]:
# --- Configuration ---

# IO paths
MANIFEST_PATH = "/data/YamaguchiLab/home_wachternl/humanGastruloid/data/metadata/metadata.tsv"
MARKERS_PATH = "/data/YamaguchiLab/home_wachternl/humanGastruloid/data/metadata/human_marker_genes.tsv"
RESULTS_DIR = "/data/YamaguchiLab/HumanGastruloidPython/results/python/09_18_2026"
PLOTS_DIR = f"{RESULTS_DIR}/plots/scvi/low_integration_strength"
OBJS_DIR_SHARED = f"{RESULTS_DIR}/objs/python"
OBJS_DIR_FINAL = f"{RESULTS_DIR}/objs/python/scvi/low_integration_strength"
PROCESSED_ADATA_PATH = f"{OBJS_DIR_SHARED}/anndata_obj_preprocessed_filtered.h5ad"
NORMALIZED_ADATA_PATH = f"{OBJS_DIR_SHARED}/anndata_obj_normalized.h5ad"
INTEGRATED_ADATA_PATH = f"{OBJS_DIR_FINAL}/anndata_obj_integrated.h5ad"
INTEGRATED_MODULE_ADATA_PATH = f"{OBJS_DIR_FINAL}/anndata_obj_integrated_with_modules.h5ad"
MOUSE_BIOMART_FILE = "/data/YamaguchiLab/home_wachternl/humanGastruloid/data/metadata/genes_xy.txt"

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(OBJS_DIR_SHARED, exist_ok=True)
os.makedirs(OBJS_DIR_FINAL, exist_ok=True)

# Variables
VARS_TO_REGRESS = ['pct_counts_mt', 'S_score', 'G2M_score']

# Curated human cell cycle genes
s_genes_human = ['MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG', 'GINS2', 'MCM6', 'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP', 'HELLS', 'RFC2', 'RPA2', 'NASP', 'RAD51AP1', 'GMNN', 'WDR76', 'SLBP', 'CCNE2', 'UBR7', 'POLD3', 'MSH2', 'ATAD2', 'RAD51', 'RRM2', 'CDC45', 'CDC6', 'EXO1', 'TIPIN', 'DSCC1', 'BLM', 'CASP8AP2', 'USP1', 'CLSPN', 'POLA1', 'CHAF1B', 'BRIP1', 'E2F8']
g2m_genes_human = ['HMGB2', 'CDK1', 'HN1', 'CDC20', 'TOP2A', 'NDC80', 'CKS2', 'NCL', 'CKS1B', 'MKI67', 'TMPO', 'CENPF', 'TACC3', 'FAM64A', 'SMC4', 'CCNB2', 'CKAP2L', 'CKAP2', 'AURKB', 'BUB1', 'KIF11', 'ANP32E', 'TUBB4B', 'GTSE1', 'KIF20B', 'HJURP', 'CDCA3', 'HN1', 'JPT1', 'CDC25C', 'KIF2C', 'RANGAP1', 'NCAPD2', 'DLGAP5', 'CDCA2', 'CDCA8', 'ECT2', 'KIF23', 'HMMR', 'AURKA', 'PSRC1', 'ANLN', 'LBR', 'CKAP5', 'CENPE', 'CTCF', 'NEK2', 'G2E3', 'GAS2L3', 'CBX5', 'CENPA']

# Overwrite objects?
OVERWRITE = True

# Chicken orthologs
#s_genes_mouse_map = scrnaseq.get_orthologs(s_genes_human, target_species="mouse")
#g2m_genes_mouse_map = scrnaseq.get_orthologs(g2m_genes_human, target_species="mouse")

# The result is a dictionary, so just get the values (the chick gene symbols)
#s_genes_mouse = list(s_genes_mouse_map.values())
#g2m_genes_mouse = list(g2m_genes_mouse_map.values())

print(f"\nExample S-phase human genes: {s_genes_human[:5]}")
print(f"Example G2/M-phase human genes: {g2m_genes_human[:5]}")

In [ ]:
# =============================================================================
# LOAD DATA (with Caching)
# =============================================================================

if os.path.exists(PROCESSED_ADATA_PATH) & OVERWRITE == False:
    # If the file exists, just load it! (This is fast)
    print(f"Loading cached processed data from: {PROCESSED_ADATA_PATH}")
    adata = sc.read_h5ad(PROCESSED_ADATA_PATH)
else:
    # If it doesn't exist, run the full pipeline...
    print("Cached file not found. Running the full loading and preprocessing pipeline...")
    
    # Run the loading workflow
    adata_raw = scrnaseq.load_and_preprocess_from_manifest(
        manifest_path=MANIFEST_PATH,
        min_genes=200,
        min_cells_per_gene=3,
        max_pct_mito=10.0,
        filter_outliers_nmads=3.0
    )
    
    # Plot QC before filtering
    scrnaseq.plot_qc_metrics(adata_raw, save_prefix=f"{PLOTS_DIR}/qc_plots_human_before_filtering")

    scrnaseq.plot_grouped_violin(
        adata_raw,
        metrics=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
        group_by='SampleID',
        stripplot=False,
        show_median=True,
        save_prefix=f"{PLOTS_DIR}/qc_human_grouped_violins_before_filtering"
    )

    # Run outlier filtering
    adata = scrnaseq.filter_outlier_cells(
        adata=adata_raw,
        library_key='SampleID',
        qc_metrics=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mt'],
        nmads=2.0
    )
    
    # Plot QC after filtering
    scrnaseq.plot_qc_metrics(adata, save_prefix=f"{PLOTS_DIR}/qc_plots_human_after_filtering")

    scrnaseq.plot_grouped_violin(
        adata,
        metrics=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
        group_by='SampleID',
        stripplot=False,
        show_median=True,
        save_prefix=f"{PLOTS_DIR}/qc_human_grouped_violins_after_filtering"
    )

    # Store raw counts before saving
    print("Storing raw counts in .layers['counts']")
    adata.layers['counts'] = adata.X.copy()
    
    # save object
    print(f"Saving processed data to: {PROCESSED_ADATA_PATH}")
    os.makedirs(OBJS_DIR_SHARED, exist_ok=True)
    adata.write_h5ad(PROCESSED_ADATA_PATH)

# --- From this point on, your 'adata' object is ready instantly on subsequent runs ---
print("\nData is ready for analysis.")
print("\nMerged object structure is as follows:\n")
adata

In [ ]:
if os.path.exists(NORMALIZED_ADATA_PATH) & OVERWRITE == False:
    print(f"Loading cached processed data from: {NORMALIZED_ADATA_PATH}")
    adata_hvg = sc.read_h5ad(NORMALIZED_ADATA_PATH)
else:
    # =============================================================================
    # SCORE CELL CYCLE AND DEFINE REGRESSION VARIABLES
    # =============================================================================
    # First, normalize the data to prepare for cell cycle scoring
    sc.pp.normalize_total(adata, target_sum=1e4)
#    sc.pp.log1p(adata)

    # Now, run the scoring function on the log-normalized data
    adata = scrnaseq.score_cell_cycle(adata, s_genes=s_genes_human, g2m_genes=g2m_genes_human, gene_symbol_col="gene_symbol")

    # =============================================================================
    # PREPROCESS FOR PCA (Regress & Scale)
    # =============================================================================
    # This function selects HVGs, regresses, and scales the data.
    adata_hvg = scrnaseq.preprocess_for_pca(adata, regress_vars=VARS_TO_REGRESS)
    # =============================================================================
    # RUN PCA
    # =============================================================================
    print("Running PCA...")
    sc.tl.pca(adata_hvg, n_comps=50, svd_solver="arpack") # Use more PCs for integration input

    print(f"Saving normalized data to: {NORMALIZED_ADATA_PATH}")
    os.makedirs(OBJS_DIR_SHARED, exist_ok=True)
    adata_hvg.write_h5ad(NORMALIZED_ADATA_PATH)


In [ ]:
print("\nMerged processed object structure is as follows:\n")
adata

In [ ]:
print("\nMerged processed object structure is as follows:\n")
adata_hvg

In [ ]:
# =============================================================================
# SCVI INTEGRATION
# =============================================================================
if os.path.exists(INTEGRATED_ADATA_PATH) & OVERWRITE == False:
    # If the file exists, just load it
    print(f"Loading cached integrated data from: {INTEGRATED_ADATA_PATH}")
    adata = sc.read_h5ad(INTEGRATED_ADATA_PATH)# Store raw counts

else:
    adata.layers["counts"] = adata.X.copy()

    # Select highly variable genes
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=3000,
        flavor="seurat_v3",
#        batch_key="SampleID",
        layer="counts",
        subset=True
    )
    
    # Register data
    scvi.model.SCVI.setup_anndata(
        adata,
        layer="counts",
        batch_key="SampleID"
    )
    
    # Train model
    model = scvi.model.SCVI(
        adata,
        n_layers=1,
        n_latent=10,
        gene_likelihood="nb"
    )
    
    model.train()
    
    # Integrated representation
    adata.obsm["X_scVI"] = model.get_latent_representation()
    
    # Downstream analysis
    print(f"Computing neighbors")
    sc.pp.neighbors(adata, use_rep="X_scVI")
    print(f"Computing UMAP coordinates")
    sc.tl.umap(adata)
    print(f"Computing clusters")
    sc.tl.leiden(adata)
    print(f"Saving integrated adata object at: {INTEGRATED_ADATA_PATH}")
    adata.write_h5ad(INTEGRATED_ADATA_PATH)


In [ ]:
# --- Visualization (Integration) ---
fig = scrnaseq.plot_umap_grid(
    adata,
    color_keys=["leiden", "SampleID", "Genotype", "Treatment"],
    ncols=2,          # Enforces a 2x2 grid
    wspace=0.5,       # Increase this value if legends still overlap
    frameon=False,    # Example of passing a kwarg to scanpy
    point_size=15
)

# Save the beautifully formatted figure
fig.savefig(f"{PLOTS_DIR}/umap_overview_grid.png", dpi=300, bbox_inches="tight")

# --- The split_umap call remains the same for the other figure ---
fig_split = scrnaseq.split_umap(adata, split_by="SampleID", color="leiden")
fig_split.savefig(f"{PLOTS_DIR}/umap_split.png", dpi=300, bbox_inches="tight")

In [ ]:
# =============================================================================
# LOAD MARKER GENES
# =============================================================================
print("--- Getting Marker Genes ---")
markers_df = pd.read_csv(MARKERS_PATH, sep="\t")
markers_by_ct = scrnaseq.markers_df_to_dict(markers_df)

# =============================================================================
# COMPUTE MODULE SCORES
# =============================================================================
modules_to_score = {
    'Alantois': markers_by_ct['Alantois'],
    'Anterior_Mesoderm': markers_by_ct['Anterior_Mesoderm'],
    'Anterior_Neuroectoderm': markers_by_ct['Anterior_Neuroectoderm'],
    'Caudolateral_Epiblast': markers_by_ct['Caudolateral_Epiblast'],
    'Endoderm_Hindgut': markers_by_ct['Endoderm_Hindgut'],
    'Lateral_Plate_Mesoderm': markers_by_ct['Lateral_Plate_Mesoderm'],
    'Neural_Tube': markers_by_ct['Neural_Tube'],
    'Notochord': markers_by_ct['Notochord'],
    'Presomitic_Mesoderm': markers_by_ct['Presomitic_Mesoderm'],
    'Primitive_Streak': markers_by_ct['Primitive_Streak'],
}

adata = scrnaseq.score_gene_modules(
    adata,
    gene_lists=modules_to_score
)

In [ ]:
# =============================================================================
# PLOT THE MODULE SCORES
# =============================================================================

# --- A) As Feature Plots on the UMAP ---
print("\n--- Plotting module scores as feature plots ---")
umap = sc.pl.umap(
    adata,
    color=['Alantois',
           'Anterior_Mesoderm',
           'Anterior_Neuroectoderm',
           'Caudolateral_Epiblast',
           'Endoderm_Hindgut',
            'Lateral_Plate_Mesoderm',
            'Neural_Tube',
            'Notochord',
            'Presomitic_Mesoderm',
            'Primitive_Streak',
          ],
    cmap='viridis', # A good colormap for scores
    return_fig=True,
#    save="_integration_seurat_best_parameters_marker_genes_featureplot.png"
)
umap.savefig(f"{PLOTS_DIR}/feature_plot_modules.png", dpi=300, bbox_inches="tight")

# --- B) As a Dot Plot ---
# This shows the average score for each gene module per cluster.
print("\n--- Plotting marker_genes as a dot plot by cluster ---")
dotplot=sc.pl.dotplot(
    adata,
    var_names=['Alantois',
           'Anterior_Mesoderm',
           'Anterior_Neuroectoderm',
           'Caudolateral_Epiblast',
           'Endoderm_Hindgut',
            'Lateral_Plate_Mesoderm',
            'Neural_Tube',
            'Notochord',
            'Presomitic_Mesoderm',
            'Primitive_Streak'], # The names are taken from the .obs columns
    groupby='leiden',
    return_fig=True,
#    save="_integration_seurat_best_parameters_marker_genes_module_score_dotplot.png"
)

dotplot.savefig(f"{PLOTS_DIR}/dotplot_modules.png", dpi=300, bbox_inches="tight")


In [ ]:
# =============================================================================
# PLOT CLUSTER PROPORTIONS ACROSS SAMPLES
# =============================================================================

print("\n--- Plotting cluster proportions by sample ---")
scrnaseq.plot_proportions(
    adata=adata,
    group_by="SampleID",
    category_to_plot="leiden",
    save_path=f"{PLOTS_DIR}/cluster_proportions_by_sample.png"
)

In [ ]:
# =============================================================================
# MARKER GENE ANALYSIS - SCVI
# =============================================================================

markers_df = pd.read_csv(MARKERS_PATH, sep="\t")
markers_by_ct = scrnaseq.markers_df_to_dict(markers_df)

# remove genes not found in the adata object
markers_by_ct_filtered = {
    ct: [g for g in genes if g in adata.var["gene_symbol"].values]
    for ct, genes in markers_by_ct.items()
}

# remove cell types with no expressed marker genes
markers_by_ct_filtered = {
    ct: genes
    for ct, genes in markers_by_ct_filtered.items()
    if len(genes) > 0
}

# Create a dotplot for the integrated results
dp = sc.pl.dotplot(
    adata,
    var_names=markers_by_ct_filtered,
    groupby="leiden",
    gene_symbols="gene_symbol",
    return_fig=True
)
dp.savefig(f"{PLOTS_DIR}/dotplot_marker_genes.png", dpi=300, bbox_inches="tight")

# UMAPs of marker genes
scrnaseq.plot_umap_markers_per_celltype(
    adata=adata,
    markers_by_celltype=markers_by_ct_filtered,
    basis="umap",                     # The embedding to use
    use_gene_symbols=True,            # Tell the function you are providing symbols, not IDs
    point_size=10,                    # Adjust point size for visibility
    ncols=4,                          # Arrange plots in a grid with 4 columns
    cmap="magma",                     # Use a different color map for expression
    
    # --- Arguments for Saving Files ---
    save_dir=PLOTS_DIR,               # The directory to save the output files
    fig_root_name="umap_marker_genes",  # A prefix for the output filenames
    combine_figures=True
)

In [ ]:
# Save final object for downstream analyses

if os.path.exists(INTEGRATED_MODULE_ADATA_PATH) & OVERWRITE == False:
    # If the file exists, just load it
    print(f"Final object already saved.")
else:
    adata.write_h5ad(INTEGRATED_MODULE_ADATA_PATH)

In [ ]:
adata

# =============================================================================
# Save images to PDF
# =============================================================================

from PIL import Image
import glob

Figs_dir="/data/YamaguchiLab/Nathaniel/humanGastruloid/results/python/07_13_2026_1/plots/scvi/low_integration_strength/"

# List your specific files in the order you want them in the PDF
png_files = [
    Figs_dir+'umap_overview_grid.png',
    Figs_dir+'umap_split.png',
    Figs_dir+'feature_plot_modules.png',
    Figs_dir+'dotplot_modules.png',
    Figs_dir+'cluster_proportions_by_sample.png',
    Figs_dir+'umap_marker_genes.all_celltypes_combined.png',
    Figs_dir+'dotplot_marker_genes.png' # from scrnaseq.plot_proportions()
]

images = [Image.open(f).convert('RGB') for f in png_files]

images[0].save(
    '/data/YamaguchiLab/Nathaniel/humanGastruloid/results/python/07_13_2026_1/pdfs/'+'scvi_low_integration_strength.pdf',
    save_all=True,
    append_images=images[1:]
)